# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant


## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and pretty-print the metadata
metadata_json = dataset.metadata.to_json()
title = dataset.metadata.name
desc = dataset.metadata.description
print(f"{title}: {desc}")

## 2. Data Overview
Review available record sets, fields, their `@id`s, and data structure.

`mlcroissant` exposes the schema entities and their attributes. We will inspect available RecordSets and list their Fields and Columns, always referencing entities by their `@id`.

In [ ]:
print("Available RecordSets and their Fields/Columns:")
print("---")
record_set_ids = [r['@id'] for r in dataset.metadata.to_json().get('recordSet', [])]

if not record_set_ids:
    print("No record sets listed explicitly in top-level metadata. Attempting to infer from distribution or columns...")
    # Try to infer RecordSets
    recs = list(dataset.metadata._json.get('distribution', []))
    if recs:
        for rec in recs:
            print(f"Distribution '@id': {rec['@id']}")
        # Use first distribution as likely tabular record set
        main_record_set_id = recs[0]['@id']
        record_set_ids = [main_record_set_id]
    else:
        print("Could not find record sets or tabular files in schema.")
else:
    print(f"Found record set `@id`s: {record_set_ids}")

# Let's print some sample records from the main record set.
for rs_id in record_set_ids:
    print(f"\nSample records for RecordSet '@id': {rs_id}")
    try:
        cnt = 0
        for rec in dataset.records(record_set=rs_id):
            pprint.pprint(rec)
            cnt += 1
            if cnt >= 2:
                break
    except Exception as e:
        print(f"Error retrieving records for {rs_id}: {e}")

## 3. Data Extraction
Load all data from the main record set into a pandas DataFrame for further analysis.

We use the record set and field/column `@id`s found in the previous step.

In [ ]:
# For this dataset, the main record set comes from the main distribution file.
# Use the record_set_ids found earlier.
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading all records from record set '@id': {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Columns for '{rs_id}': {df.columns.tolist()}")
        print(f"\nPreview of DataFrame for record set '@id': {rs_id}")
        display(df.head())
    else:
        print(f"No records found for record set {rs_id}.")

# For following steps, pick the first record set as main
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]
fields_available = df.columns.tolist()

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, categorizing data, and basic aggregations.

*Note: All field references (columns) should use their `@id` as per schema.*

In [ ]:
# List numeric fields (by inferring from pandas dtypes)
numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

print(f"Numeric fields detected: {numeric_cols}")

if numeric_cols:
    # Choose the first numeric field for demonstration
    numeric_field = numeric_cols[0]  # e.g., '@id' of a numeric column
    print(f"Using numeric_field = '{numeric_field}'")
    
    # Set an arbitrary threshold at the field's mean
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean value):\n")
    display(filtered_df.head())
    
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())
    
    # Find a likely grouping field (object dtype, not the index or numeric)
    group_candidates = [c for c in df.columns if pd.api.types.is_object_dtype(df[c]) and c != numeric_field]
    if group_candidates:
        group_field = group_candidates[0]
        print(f"Grouping by field '{group_field}'\n")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index(name=f"mean_{numeric_field}")
        display(grouped_df.head())
    else:
        print("No suitable object/group field found for grouping.")
else:
    print("No numeric fields found in record set for EDA.")

## 5. Visualization
Visualize data distributions or relationships using matplotlib/seaborn.

Here, we plot a histogram of the chosen numeric field, and if a grouping field was found, a bar chart of means by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_cols:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    if 'group_field' in locals():
        plt.figure(figsize=(8,4))
        sns.barplot(data=grouped_df, x=group_field, y=f"mean_{numeric_field}")
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
In this notebook, we've used the `mlcroissant` library to load and explore the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset. We loaded metadata, listed available record sets and fields using `@id` references, and performed basic exploratory analysis including filtering, normalization, grouping, and visualizations. This workflow can be further extended for domain-specific downstream analysis.